# KIỂM TRA VẬN HÀNH ĐƯỜNG BĂNG
**Mục tiêu:** Phát hiện các Data Anomalies trái với quy tắc vật lý bay và quy trình hàng không thực tế.

## Quy tắc vận hành đường băng
1. **Hướng đường băng (Runway Heading):**
   - SGN: 07L/25R, 07R/25L (Heading: 070°/250°)
   - HAN: 11L/29R, 11R/29L (Heading: 110°/290°)
   - DAD: 17L/35R, 17R/35L (Heading: 170°/350°)
2. **Ngưỡng gió xuôi an toàn (Tailwind Limit):** 
   - Theo quy chuẩn an toàn, gió xuôi > 15 knots (27.8 km/h) là giới hạn tuyệt đối. Mọi bản ghi cất cánh vượt ngưỡng này đều là **Dữ liệu lỗi** (cào nhầm hướng đường băng).
3. **Chỉ số Gió giật (Gust Factor):** 
   - Gió giật > 25 knots (46.3 km/h) gây nhiễu động nghiêm trọng (Wind Shear), buộc phải ghi nhận vào Audit.

In [3]:
import os
import pandas as pd
import numpy as np
import math
from pathlib import Path

# 1. Hàm định hướng đường băng
def get_runway_heading(rw_str):
    if pd.isna(rw_str): return np.nan
    rw_str = str(rw_str).upper()
    if '07' in rw_str: return 70
    if '25' in rw_str: return 250
    if '11' in rw_str: return 110
    if '29' in rw_str: return 290
    if '17' in rw_str: return 170
    if '35' in rw_str: return 350
    return np.nan

# 2. Hàm tính Gió xuôi (Tailwind) theo lượng giác
def calculate_tailwind_math(row):
    rw_val = row['Departure_Runway'] if row['Record_Type'] == 'Departure' else row['Arrival_Runway']
    rw_heading = get_runway_heading(rw_val)

    if pd.isna(rw_heading) or pd.isna(row['wind_speed']) or pd.isna(row['wind_direction']):
        return 0

    # Công thức: Component = Spd * cos(Gió - Băng)
    angle_diff_rad = math.radians(row['wind_direction'] - rw_heading)
    wind_comp = row['wind_speed'] * math.cos(angle_diff_rad)
    return abs(wind_comp) if wind_comp < 0 else 0

## Bước thực thi Audit
Script quét toàn bộ file trong Silver Layer và đối chiếu với Weather Data.

In [4]:
def run_full_audit(project_root=r"/Users/nguyenhung/PycharmProjects/DS108_AeroDelay"):
    project_root = Path(project_root)
    silver_path = project_root / "Data" / "Silver_layer"
    weather_path = project_root / "Data" / "Silver_layer" / "Features" / "weather_features_hourly.csv"

    df_weather = pd.read_csv(weather_path)
    df_weather['time'] = pd.to_datetime(df_weather['time'], errors='coerce')
    
    all_anomalies = []

    for apt in ["sgn", "han", "dad"]:
        for cat in ["Departure", "Arrival"]:
            file_path = silver_path / cat / f"{apt}_flights_{cat.lower()}_gold_layer.csv"
            if not file_path.exists(): continue

            df = pd.read_csv(file_path)
            df['Actual_Time'] = pd.to_datetime(df['Actual_Time'], errors='coerce')
            df = df.dropna(subset=['Actual_Time'])

            # Merge thời tiết
            df_w = df_weather[df_weather['Airport'] == apt.upper()].sort_values('time')
            df = df.sort_values('Actual_Time')
            df_audit = pd.merge_asof(df, df_w, left_on='Actual_Time', right_on='time', direction='nearest')

            # Tính Tailwind
            df_audit['Tailwind_Kmh'] = df_audit.apply(calculate_tailwind_math, axis=1)

            # Xác định các ca Lỗi vật lý (Anomaly detection)
            df_audit['Impossible_Tailwind'] = df_audit['Tailwind_Kmh'] > 27.8
            df_audit['Extreme_Gust'] = df_audit['Wind_Gust_Estimate_Kmh'] > 46.3
            
            # Thu thập mẫu lỗi
            anomalies = df_audit[df_audit[['Impossible_Tailwind', 'Extreme_Gust']].any(axis=1)].copy()
            if not anomalies.empty:
                anomalies['Source_File'] = file_path.name
                all_anomalies.append(anomalies)

    # Xuất kết quả
    if all_anomalies:
        final_audit = pd.concat(all_anomalies, ignore_index=True)
        final_audit.to_csv(silver_path / "Audit" / "audit_runway_anomalies_all.csv", index=False)
        print(f"[*] Kiểm toán hoàn tất. Tìm thấy {len(final_audit)} mẫu lỗi.")
    else:
        print("[V] Hệ thống hoàn toàn sạch (0 lỗi).")

run_full_audit()

[V] Hệ thống hoàn toàn sạch (0 lỗi).
